# 🔎 VeriFile AI — Transfer Screenshot & Document Forensics

**Mode 1 (tanpa GPU):** analisis metadata / PDF / DOCX / JPG, deteksi penimpaan teks PDF (paint-order), deteksi area nominal RapidOCR, visualisasi tekstur lokal OpenCV, diagnostik foto ulang layar HP, dan perbandingan piksel terhadap referensi transaksi asli + laporan JSON. **Mode 2 (opsional GPU):** Gemma 4 E2B multimodal untuk membaca dan mendeskripsikan observasi visual. Tidak bisa memastikan transaksi asli, penipuan, atau keabsahan transfer secara mutlak.

**Cara memakai:** Buka notebook ini di [Google Colab](https://colab.research.google.com/) → *Runtime → Change runtime type → T4 GPU* (opsional untuk Gemma 4) → jalankan cell secara berurutan. Mode forensik inti berjalan penuh tanpa memerlukan GPU.

⚠️ Untuk keamanan privasi, gunakan data sintetis atau dokumen berizin. Demo publik Gradio tidak disarankan untuk data keuangan pribadi riil.

In [ ]:
import os, sys, shutil, subprocess
print('Python:', sys.version.split()[0])
try:
    import torch
    print('GPU aktif:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('Device:', torch.cuda.get_device_name(0))
except ImportError:
    print('Torch belum diinstal; mode forensik tetap bisa digunakan.')

## 1. Clone repositori VeriFile-AI

Clone kode langsung dari repositori GitHub `https://github.com/alifakbxr/VeriFile-AI.git`. Tidak perlu mengunggah ZIP secara manual.

In [ ]:
## 2. Instal dependensi, OCR dan uji forensic engine

Colab mengunduh model RapidOCR/ONNX Runtime secara otomatis. Menjalankan penyiapan sampel sintetis, suite uji regresi lengkap (27 pengujian termasuk validasi keamanan DoS/XML/PII), serta ekspor laporan forensik CLI.

## 2. Instal dependensi, OCR dan uji forensic engine

Colab memiliki akses internet untuk mengunduh RapidOCR/ONNX Runtime. Integrasi OCR di bundel ini belum teruji karena jaringan lingkungan pembuat tidak tersedia. Jika gagal, gunakan `requirements.txt` dan mode tanpa OCR, atau lakukan perbaikan berdasarkan traceback.


In [ ]:
%pip -q install -r requirements-ocr.txt
!python demo/create_samples.py
!python -m pytest -q
!python -m verifile.cli demo/edited_overlay_sample.pdf --output demo/report_demo.json


## 3. Opsional: aktifkan Gemma 4 multimodal di Colab GPU

Model default `google/gemma-4-E2B-it`. Gunakan *T4 GPU* atau lebih baik. Jika Colab GPU sedang tidak tersedia, **lewati cell ini** dan lanjutkan UI. Pilihan Gemma pada UI dalam keadaan OFF. 4-bit bisa mengurangi memori GPU, tetapi bukan jaminan kompatibilitas. Sebelum download model, pastikan storage Colab cukup; bobot model standar sekitar 10 GB.

In [ ]:
# SET true hanya setelah GPU aktif dan Anda siap mengunduh model Hugging Face.
ENABLE_GEMMA = False
if ENABLE_GEMMA:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Pilih T4 GPU di Runtime > Change runtime type terlebih dahulu.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-gemma.txt'], check=True)
    os.environ['GEMMA_4BIT'] = '1'
    os.environ['GEMMA_MODEL_ID'] = 'google/gemma-4-E2B-it'
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')  # Colab > Secrets; jangan disimpan di notebook.
        if hf_token:
            from huggingface_hub import login
            login(token=hf_token, add_to_git_credential=False)
            del hf_token
            print('Hugging Face authentication configured.')
    except Exception as e:
        print('Colab secret HF_TOKEN belum diset (jika model membutuhkan akses, set token):', type(e).__name__)
    print('Gemma siap di-load ketika checkbox UI ON. Loading pertama dapat memerlukan download besar.')
else:
    print('Skip Gemma. Forensic engine dapat berjalan tanpa GPU.')

## 4. Jalankan demo Gradio

Untuk melihat app dari browser di Colab, link share dapat dipakai; **jangan upload dokumen finansial nyata** pada demo publik. Bila share tidak dipilih, jalankan dengan fasilitas proxy Colab (jika tersedia). Setelah selesai, tutup tab dan matikan runtime.

In [ ]:
# Boleh true HANYA untuk contoh sintetis; link publik berpotensi diakses pihak lain.
SHARE_PUBLIC_DEMO = False

# Konfigurasi autentikasi opsional jika share=True (username & password)
import os
# os.environ["VERIFILE_AUTH_USER"] = "admin"
# os.environ["VERIFILE_AUTH_PASS"] = "hacktoberfest2026"

auth = None
if os.getenv("VERIFILE_AUTH_USER") and os.getenv("VERIFILE_AUTH_PASS"):
    auth = (os.environ["VERIFILE_AUTH_USER"], os.environ["VERIFILE_AUTH_PASS"])

import app as verifile_app
verifile_app.build_app().launch(
    server_name="0.0.0.0",
    share=SHARE_PUBLIC_DEMO,
    auth=auth,
    debug=True,
    max_file_size="20mb"
)

## Checklist demo hack day

1. Run tests (`27 passed` setelah audit keamanan & mitigasi celah DoS / XML / PII; teruji di Colab).
2. Compare `demo/edited_sample.jpg` with `demo/clean_sample.jpg` as the **same-transaction original**; inspect highlighted differences.
3. Upload `demo/edited_overlay_sample.pdf` and inspect the `pdf_text_cover_replace` finding: the file has retained text → patch → replacement text paint order.
4. Use RapidOCR highlights and the local-texture heatmap as *review cues*, **not as proof** of fraud or AI generation. The first OCR invocation may initialize models.
5. If GPU and sufficient disk space exist, enable Gemma 4 and review full screenshot + crops of probable amount areas.
6. Explain that only an official bank ledger/authorized API can verify transfer settlement.

Event: https://events.mlh.com/events/15226  |  Gemma docs: https://ai.google.dev/gemma/docs/core/huggingface_inference

7. Upload `demo/screen_photo_illustration.jpg`: inspect `forensics.recapture_diagnostics` and the candidate outline. It is synthetic artwork, not a photograph, and does not validate real-world recapture detection. Real camera date refers to capture time, not transaction time.